# Nivel 4: la caja grande de los pingüinos

**Nivel:** intermediate

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jcval94/narrative/blob/codex/corporate-data-narrative-lab/corporate-data-narrative-lab/outputs/notebooks/65-la-caja-grande-de-los-pinguinos.ipynb)

## Pregunta central

¿Podemos usar la profundidad del pico para decidir qué pingüinos necesitan una caja de transporte grande?

## Recreación narrativa

> **Memo:** "La base dice que los picos profundos pesan menos. Pidan 140 cajas chicas."
>
> **Dalia:** "¿Separaste las especies?"
>
> **Memo:** "Compras cotiza por caja, no por zoología. La orden sale a las cuatro."
>
> **La vicepresidenta:** "Y si no caben, el cambio tarda seis semanas. Abre los puntos."

*La escena es una recreación; las conclusiones provienen del dataset citado.*

## Fuente real

**Palmer Penguins (penguins)**, Allison Horst, Alison Hill y Kristen Gorman; datos de Palmer Station LTER. [Página de origen](https://allisonhorst.github.io/palmerpenguins/) · [datos](https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv) · licencia: CC0.  
Consultado: 2026-07-21 · 344 filas · columnas usadas: `species`, `island`, `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`, `sex`, `year`.

In [1]:
# @title Preparar los datos { display-mode: "form" }
anio = "Todos" # @param ["Todos", "2007", "2008", "2009"]
import io
import hashlib
import urllib.request
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display

DATA_URL = "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv"
EXPECTED_SHA256 = "f204db2c753b0937caac3cb35258562c14f073e4bbc76be24b4c51ce22767a93"
request = urllib.request.Request(DATA_URL, headers={"User-Agent": "Mozilla/5.0"})
try:
    with urllib.request.urlopen(request, timeout=30) as response:
        csv_bytes = response.read()
except Exception as exc:
    raise RuntimeError(f"No se pudo descargar Palmer Penguins. Revisa {DATA_URL}") from exc
assert hashlib.sha256(csv_bytes).hexdigest() == EXPECTED_SHA256, "La fuente cambió; vuelve a verificar el CSV."
df = pd.read_csv(io.BytesIO(csv_bytes))
filas_fuente = len(df)
faltantes_clave = int(df[["bill_depth_mm", "body_mass_g"]].isna().any(axis=1).sum())
df = df.dropna(subset=["bill_depth_mm", "body_mass_g"]).copy()
df["sex"] = df["sex"].fillna("Sin dato")
if anio != "Todos":
    df = df.loc[df["year"].eq(int(anio))].copy()

df["pico_profundo_global"] = df["bill_depth_mm"].ge(18)
df["masa_alta_global"] = df["body_mass_g"].ge(4000)
df["mediana_pico_especie"] = df.groupby("species")["bill_depth_mm"].transform("median")
df["mediana_masa_especie"] = df.groupby("species")["body_mass_g"].transform("median")
df["pico_profundo_relativo"] = df["bill_depth_mm"].ge(df["mediana_pico_especie"])
df["masa_alta_relativa"] = df["body_mass_g"].ge(df["mediana_masa_especie"])

def resumen_2x2(tabla):
    riesgo = tabla[True].div(tabla.sum(axis=1))
    odds = tabla[True].div(tabla[False])
    return pd.Series({"P(alta|profundo)": riesgo.loc[True], "P(alta|no profundo)": riesgo.loc[False],
                      "riesgo relativo": riesgo.loc[True]/riesgo.loc[False], "odds ratio": odds.loc[True]/odds.loc[False]})
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year,pico_profundo_global,masa_alta_global,mediana_pico_especie,mediana_masa_especie,pico_profundo_relativo,masa_alta_relativa
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007,True,False,18.4,3700.0,True,True
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007,False,False,18.4,3700.0,False,True
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007,True,False,18.4,3700.0,False,False
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007,True,False,18.4,3700.0,True,False
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,male,2007,True,False,18.4,3700.0,True,False


## 1. Relación visual

> **Dalia:** "La línea negra baja, pero los colores suben."
>
> **Memo:** "No me encanta que la misma gráfica me contradiga en dos idiomas."


**Pregunta:** ¿Qué forma tiene la relación entre profundidad del pico y masa corporal?

**Conexión:** Punto de partida: ver tendencia, forma y grupos antes de resumir.

In [2]:
resumen_visual = df.groupby("species").agg(
    pingüinos=("species", "size"),
    pico_mediano_mm=("bill_depth_mm", "median"),
    masa_mediana_g=("body_mass_g", "median")
)
resumen_visual

,pingüinos,pico_mediano_mm,masa_mediana_g
species,,,
Adelie,151,18.40,3700.0
Chinstrap,68,18.45,3700.0
Gentoo,123,15.00,5000.0


In [3]:
# @title Explorar Relación visual { display-mode: "form" }
especies = ["Adelie", "Chinstrap", "Gentoo"]
colores = {"Adelie": "#0072B2", "Chinstrap": "#D55E00", "Gentoo": "#009E73"}
simbolos = {"Adelie": "circle", "Chinstrap": "diamond", "Gentoo": "triangle-up"}
fig = go.Figure()
fig.add_trace(go.Scatter(x=df["bill_depth_mm"], y=df["body_mass_g"], mode="markers", name="Todos sin separar",
                         marker={"color": "#A7B0BE", "size": 8, "opacity": .55}, visible=False,
                         customdata=np.column_stack([df["species"], df["sex"], df["island"]]),
                         hovertemplate="Pico=%{x:.1f} mm<br>Masa=%{y:,.0f} g<br>%{customdata[0]} · %{customdata[1]}<br>Isla=%{customdata[2]}<extra></extra>"))
x_total = np.linspace(df["bill_depth_mm"].min(), df["bill_depth_mm"].max(), 80)
m_total, b_total = np.polyfit(df["bill_depth_mm"], df["body_mass_g"], 1)
fig.add_trace(go.Scatter(x=x_total, y=m_total*x_total+b_total, mode="lines", name="Tendencia agregada",
                         line={"color": "#2F3640", "width": 4, "dash": "dot"},
                         hovertemplate="Agregado<br>Pico=%{x:.1f} mm<br>Masa estimada=%{y:,.0f} g<extra></extra>"))
for especie in especies:
    grupo = df.loc[df["species"].eq(especie)]
    fig.add_trace(go.Scatter(x=grupo["bill_depth_mm"], y=grupo["body_mass_g"], mode="markers", name=especie,
                             marker={"color": colores[especie], "symbol": simbolos[especie], "size": 9, "opacity": .72,
                                     "line": {"color": "white", "width": .7}},
                             customdata=np.column_stack([grupo["sex"], grupo["island"]]),
                             hovertemplate=f"{especie}<br>Pico=%{{x:.1f}} mm<br>Masa=%{{y:,.0f}} g<br>Sexo=%{{customdata[0]}}<br>Isla=%{{customdata[1]}}<extra></extra>"))
    x_linea = np.linspace(grupo["bill_depth_mm"].min(), grupo["bill_depth_mm"].max(), 50)
    pendiente, intercepto = np.polyfit(grupo["bill_depth_mm"], grupo["body_mass_g"], 1)
    fig.add_trace(go.Scatter(x=x_linea, y=pendiente*x_linea+intercepto, mode="lines", name=f"{especie} · tendencia",
                             line={"color": colores[especie], "width": 3}, showlegend=False,
                             hovertemplate=f"{especie} · tendencia<br>Pico=%{{x:.1f}} mm<br>Masa estimada=%{{y:,.0f}} g<extra></extra>"))
agregado = [True, True] + [False]*6; por_especie = [False, False] + [True]*6
contraste = [False, True] + [True]*6
fig.update_layout(title=f"Una línea baja; tres líneas suben<br><sup>Palmer Penguins · año: {anio} · n={len(df)} completos; {faltantes_clave} excluidos</sup>",
                  xaxis_title="Profundidad del pico (mm)", yaxis_title="Masa corporal (g)", template="plotly_white",
                  legend_title="Vista", hovermode="closest", margin={"t": 95, "r": 30},
                  updatemenus=[{"buttons": [{"label": "Contraste", "method": "update", "args": [{"visible": contraste}]},
                                              {"label": "Agregado", "method": "update", "args": [{"visible": agregado}]},
                                              {"label": "Por especie", "method": "update", "args": [{"visible": por_especie}]}],
                                "x": 1, "xanchor": "right"}])
fig.add_annotation(x=19.9, y=5250, text="Agregado: pendiente negativa", showarrow=True, arrowhead=2, bgcolor="#FFFFFF")
fig.show()


display(Markdown("**Lo que muestra:** El agregado desciende, pero las tres tendencias por especie ascienden. Los grupos ocupan zonas distintas: Gentoo concentra mayor masa y picos menos profundos, así que una sola línea mezcla composición con relación."))

**Lo que muestra:** El agregado desciende, pero las tres tendencias por especie ascienden. Los grupos ocupan zonas distintas: Gentoo concentra mayor masa y picos menos profundos, así que una sola línea mezcla composición con relación.

## 2. Correlación

> **Memo:** "Entonces dame el número negro, el que baja."
>
> **Dalia:** "Te doy Pearson, Spearman y la prueba de extremos; ninguno separa especies todavía."


**Pregunta:** ¿Pearson y Spearman sostienen la relación negativa agregada, incluso sin los extremos?

**Conexión:** Relación visual descubrió tendencias opuestas; Correlación mide dirección, fuerza y sensibilidad a extremos.

In [4]:
limites = df[["bill_depth_mm", "body_mass_g"]].quantile([.01, .99])
centrales = df["bill_depth_mm"].between(*limites["bill_depth_mm"]) & df["body_mass_g"].between(*limites["body_mass_g"])
resumen_correlacion = pd.DataFrame({
    "Pearson": [df["bill_depth_mm"].corr(df["body_mass_g"], method="pearson"), df.loc[centrales, "bill_depth_mm"].corr(df.loc[centrales, "body_mass_g"], method="pearson")],
    "Spearman": [df["bill_depth_mm"].corr(df["body_mass_g"], method="spearman"), df.loc[centrales, "bill_depth_mm"].corr(df.loc[centrales, "body_mass_g"], method="spearman")]
}, index=["Todos", "Centro 1%-99%"])
resumen_correlacion.round(3)

,Pearson,Spearman
Todos,-0.472,-0.432
Centro 1%-99%,-0.488,-0.427


In [5]:
# @title Explorar Correlación { display-mode: "form" }
escenarios = ["Todos", "Centro 1%-99%"]
fig = go.Figure()
for escenario in escenarios:
    valores = resumen_correlacion.loc[escenario]
    fig.add_trace(go.Scatter(x=[valores["Pearson"], valores["Spearman"]], y=[escenario, escenario], mode="lines",
                             line={"color": "#98A2B3", "width": 6}, hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter(x=resumen_correlacion["Pearson"], y=escenarios, mode="markers+text", name="Pearson",
                         marker={"color": "#0072B2", "size": 16, "symbol": "circle"},
                         text=[f"{v:.3f}" for v in resumen_correlacion["Pearson"]], textposition="top center",
                         hovertemplate="%{y}<br>Pearson=%{x:.3f}<extra></extra>"))
fig.add_trace(go.Scatter(x=resumen_correlacion["Spearman"], y=escenarios, mode="markers+text", name="Spearman",
                         marker={"color": "#D55E00", "size": 16, "symbol": "diamond"},
                         text=[f"{v:.3f}" for v in resumen_correlacion["Spearman"]], textposition="bottom center",
                         hovertemplate="%{y}<br>Spearman=%{x:.3f}<extra></extra>"))
fig.update_layout(title=f"La correlación negativa no depende de unos pocos extremos<br><sup>Año: {anio} · recorte simultáneo en pico y masa · n={len(df)}</sup>",
                  xaxis_title="Coeficiente de correlación (-1 a 1)", yaxis_title="Muestra analizada", xaxis_range=[-1, 1],
                  template="plotly_white", margin={"t": 95},
                  updatemenus=[{"buttons": [{"label": "Comparar", "method": "update", "args": [{"visible": [True, True, True, True]}]},
                                              {"label": "Pearson", "method": "update", "args": [{"visible": [True, True, True, False]}]},
                                              {"label": "Spearman", "method": "update", "args": [{"visible": [True, True, False, True]}]}], "x": 1, "xanchor": "right"}])
fig.add_vline(x=0, line_color="#344054", line_dash="dot")
fig.show()


display(Markdown("**Lo que muestra:** En el agregado, Pearson es -0.472 y Spearman -0.432. Tras retirar el 1% extremo de ambas variables quedan en -0.488 y -0.427: la dirección negativa es estable, pero sigue siendo una descripción de la mezcla."))

**Lo que muestra:** En el agregado, Pearson es -0.472 y Spearman -0.432. Tras retirar el 1% extremo de ambas variables quedan en -0.488 y -0.427: la dirección negativa es estable, pero sigue siendo una descripción de la mezcla.

## 3. Variables de confusión

> **La vicepresidenta:** "El negativo sobrevivió a los extremos. ¿Ya cierro la compra?"
>
> **Dalia:** "No. Al separar especies, las tres correlaciones cambian a positivas."


**Pregunta:** ¿Qué ocurre con Pearson cuando dejamos de mezclar especies?

**Conexión:** Correlación confirmó el signo agregado; Variables de confusión pregunta si especie explica ese signo.

In [6]:
corr_especie = {especie: grupo["bill_depth_mm"].corr(grupo["body_mass_g"])
                for especie, grupo in df.groupby("species")}
resumen_confusores = pd.Series({
    "Agregado": df["bill_depth_mm"].corr(df["body_mass_g"]), **corr_especie
}, name="Pearson").to_frame()
resumen_confusores.round(3)

,Pearson
Agregado,-0.472
Adelie,0.576
Chinstrap,0.604
Gentoo,0.719


In [7]:
# @title Explorar Variables de confusión { display-mode: "form" }
orden = ["Agregado", "Adelie", "Chinstrap", "Gentoo"]
colores = {"Agregado": "#B42318", "Adelie": "#0072B2", "Chinstrap": "#D55E00", "Gentoo": "#009E73"}
fig = go.Figure()
for grupo in orden:
    valor = resumen_confusores.loc[grupo, "Pearson"]
    fig.add_trace(go.Scatter(x=[0, valor], y=[grupo, grupo], mode="lines+markers+text", name=grupo,
                             line={"color": colores[grupo], "width": 7},
                             marker={"color": colores[grupo], "size": [7, 17], "symbol": "circle"},
                             text=["", f"{valor:+.3f}"], textposition="middle right" if valor >= 0 else "middle left",
                             hovertemplate=f"{grupo}<br>Pearson=%{{x:.3f}}<extra></extra>"))
fig.update_layout(title=f"La especie invierte el signo de la relación<br><sup>Pico vs. masa · año: {anio} · n={len(df)}</sup>",
                  xaxis_title="Correlación de Pearson (-1 a 1)", yaxis_title="Nivel de análisis", xaxis_range=[-.9, .95],
                  template="plotly_white", showlegend=False, margin={"t": 95, "l": 105},
                  updatemenus=[{"buttons": [{"label": "Contraste", "method": "update", "args": [{"visible": [True, True, True, True]}]},
                                              {"label": "Agregado", "method": "update", "args": [{"visible": [True, False, False, False]}]},
                                              {"label": "Especies", "method": "update", "args": [{"visible": [False, True, True, True]}]}], "x": 1, "xanchor": "right"}])
fig.add_vline(x=0, line_color="#344054", line_dash="dot")
fig.add_annotation(x=.05, y="Gentoo", text="Misma dirección dentro de cada especie", showarrow=False, xanchor="left", bgcolor="#ECFDF3")
fig.show()


display(Markdown("**Lo que muestra:** El agregado es -0.472, mientras Adelie, Chinstrap y Gentoo dan 0.576, 0.604 y 0.719. Es una inversión por agregación: la especie se relaciona tanto con la forma del pico como con la masa y confunde la comparación global."))

**Lo que muestra:** El agregado es -0.472, mientras Adelie, Chinstrap y Gentoo dan 0.576, 0.604 y 0.719. Es una inversión por agregación: la especie se relaciona tanto con la forma del pico como con la masa y confunde la comparación global.

## 4. Tablas cruzadas

> **Óscar:** "Compras necesita grande o chica, no una conferencia sobre signos."
>
> **Dalia:** "Perfecto: hagamos dos tablas y veamos cuál regla manda al pingüino equivocado."


**Pregunta:** ¿Cómo cambia la decisión al usar cortes globales frente a cortes relativos dentro de cada especie?

**Conexión:** Variables de confusión reveló una inversión de signo; Tablas cruzadas traduce ambos enfoques a proporciones, riesgo relativo y odds.

In [8]:
tabla_global = pd.crosstab(df["pico_profundo_global"], df["masa_alta_global"]).reindex(index=[False, True], columns=[False, True], fill_value=0)
tabla_especie = pd.crosstab(df["pico_profundo_relativo"], df["masa_alta_relativa"]).reindex(index=[False, True], columns=[False, True], fill_value=0)
resumen_tablas = pd.DataFrame({
    "Cortes globales": resumen_2x2(tabla_global),
    "Dentro de especie": resumen_2x2(tabla_especie)
}).T
display(tabla_global.rename_axis("Pico ≥18 mm"), tabla_especie.rename_axis("Pico ≥ mediana de especie"))
resumen_tablas.round(3)

masa_alta_global,False,True
Pico ≥18 mm,,
False,83,124
True,82,53


masa_alta_relativa,False,True
Pico ≥ mediana de especie,,
False,120,48
True,42,132


,P(alta|profundo),P(alta|no profundo),riesgo relativo,odds ratio
Cortes globales,0.393,0.599,0.655,0.433
Dentro de especie,0.759,0.286,2.655,7.857


In [9]:
# @title Explorar Tablas cruzadas { display-mode: "form" }
global_no = 100*resumen_tablas.loc["Cortes globales", "P(alta|no profundo)"]; global_si = 100*resumen_tablas.loc["Cortes globales", "P(alta|profundo)"]
especie_no = 100*resumen_tablas.loc["Dentro de especie", "P(alta|no profundo)"]; especie_si = 100*resumen_tablas.loc["Dentro de especie", "P(alta|profundo)"]
fig = go.Figure()
fig.add_trace(go.Bar(name="Pico no profundo", x=["Cortes globales"], y=[global_no], marker_color="#667085",
                     text=[f"{global_no:.1f}%"], textposition="outside", hovertemplate="Global · no profundo<br>P(masa alta)=%{y:.1f}%<extra></extra>"))
fig.add_trace(go.Bar(name="Pico profundo", x=["Cortes globales"], y=[global_si], marker_color="#B42318",
                     text=[f"{global_si:.1f}%"], textposition="outside", hovertemplate="Global · profundo<br>P(masa alta)=%{y:.1f}%<extra></extra>"))
fig.add_trace(go.Bar(name="Pico no profundo · ajustado", x=["Dentro de especie"], y=[especie_no], marker_color="#667085", showlegend=False,
                     text=[f"{especie_no:.1f}%"], textposition="outside", hovertemplate="Dentro de especie · no profundo<br>P(masa alta)=%{y:.1f}%<extra></extra>"))
fig.add_trace(go.Bar(name="Pico profundo · ajustado", x=["Dentro de especie"], y=[especie_si], marker_color="#009E73", showlegend=False,
                     text=[f"{especie_si:.1f}%"], textposition="outside", hovertemplate="Dentro de especie · profundo<br>P(masa alta)=%{y:.1f}%<extra></extra>"))
fig.update_layout(barmode="group", title=f"La regla cambia cuando los cortes respetan la especie<br><sup>Global: pico ≥18 mm y masa ≥4,000 g · relativo: medianas por especie · año: {anio}</sup>",
                  xaxis_title="Forma de construir la tabla 2×2", yaxis_title="Proporción con masa alta (%)", yaxis_range=[0, 100],
                  template="plotly_white", margin={"t": 105}, legend_title="Condición del pico",
                  updatemenus=[{"buttons": [{"label": "Comparar", "method": "update", "args": [{"visible": [True, True, True, True]}]},
                                              {"label": "Global", "method": "update", "args": [{"visible": [True, True, False, False]}]},
                                              {"label": "Dentro de especie", "method": "update", "args": [{"visible": [False, False, True, True]}]}], "x": 1, "xanchor": "right"}])
fig.add_hline(y=50, line_color="#98A2B3", line_dash="dot")
fig.add_annotation(x="Cortes globales", y=88, text=f"RR={resumen_tablas.loc['Cortes globales','riesgo relativo']:.3f}<br>OR={resumen_tablas.loc['Cortes globales','odds ratio']:.3f}", showarrow=False, bgcolor="#FEF3F2")
fig.add_annotation(x="Dentro de especie", y=88, text=f"RR={resumen_tablas.loc['Dentro de especie','riesgo relativo']:.3f}<br>OR={resumen_tablas.loc['Dentro de especie','odds ratio']:.3f}", showarrow=False, bgcolor="#ECFDF3")
fig.show()


display(Markdown("**Lo que muestra:** Con cortes globales, pico profundo parece reducir masa alta: 39.3% frente a 59.9%, RR 0.655 y OR 0.433. Con cortes relativos por especie ocurre lo contrario: 75.9% frente a 28.6%, RR 2.655 y OR 7.857. Son asociaciones descriptivas, no efectos causales."))

**Lo que muestra:** Con cortes globales, pico profundo parece reducir masa alta: 39.3% frente a 59.9%, RR 0.655 y OR 0.433. Con cortes relativos por especie ocurre lo contrario: 75.9% frente a 28.6%, RR 2.655 y OR 7.857. Son asociaciones descriptivas, no efectos causales.

## Cómo se conecta todo

El scatterplot reveló grupos; Pearson y Spearman midieron una relación negativa estable frente a extremos; la separación por especie invirtió el signo; y las tablas 2×2 mostraron que los cortes globales y los cortes dentro de especie llevan a decisiones opuestas.

## Decisión

No comprar cajas con una regla global basada en el pico. Registrar especie y masa; si falta la masa, calibrar cualquier proxy por especie y validarlo antes de usarlo en logística.

**Regla:** Cuando los grupos ocupan regiones distintas, una relación agregada puede cambiar de signo: visualiza, mide, estratifica y recién entonces convierte categorías en una regla.